# 03 · Entrenamiento RSNA Knee (Fase 4)

Settings del notebook:
- **Accelerator**: GPU T4 x2 (se usa una sola) o P100.
- **Internet**: On, para bajar los pesos ImageNet de torchvision (ResNet-18).
- **Add Input**: el Dataset privado `rsna-knee-subset-v1`.

Modelos (entregable 2):
1. `scratch_base`: CNN 2.5D propia (bloques residuales conv-BN-ReLU), sin pesos previos.
2. `resnet18_base`: ResNet-18 ImageNet; 2 épocas con el backbone congelado y luego fine-tuning con LR × 0.1.

Ambos: 16 ventanas de 3 cortes por serie, sagital y coronal, agregación por atención entre cortes,
12 sigmoides, BCE con `pos_weight`, máscara para UNC y WEAK, sin flips.
El checkpoint se elige por AUC macro en val (etiquetas débiles); los 58 dorados solo se predicen al final.

Salida en `/kaggle/working/runs/<nombre>/`: `best.pt`, `history.csv`, `preds_val.csv`, `preds_test.csv` y `metrics.json`.

In [ ]:
import glob, json, os, subprocess, sys
from pathlib import Path
import pandas as pd

def find(pattern, max_depth=4):
    for d in range(1, max_depth + 1):
        hits = sorted(glob.glob("/kaggle/input/" + "*/" * (d - 1) + pattern))
        if hits:
            return Path(hits[0])
    raise FileNotFoundError(f"no encuentro {pattern} en /kaggle/input (¿falta Add Input?)")

DATA = find("subset_studies.csv").parent
studies = pd.read_csv(DATA / "subset_studies.csv")
print(DATA, studies.split.value_counts().to_dict(), "· volúmenes:", len(list((DATA / "vol").glob("*.npy"))))
assert studies.split.value_counts().to_dict() == {"train": 400, "val": 100, "test": 58}

import torch, torchvision
print("torch", torch.__version__, "· torchvision", torchvision.__version__, "· cuda", torch.cuda.is_available())
assert torch.cuda.is_available(), "activar GPU en Settings → Accelerator"
print(torch.cuda.get_device_name(0))
RUNS = Path("/kaggle/working/runs"); RUNS.mkdir(parents=True, exist_ok=True)

## 1. Código del repo (copiado tal cual por `notebooks/build_03_train.py`)

In [ ]:
CODE = Path('/tmp/rsnak')
FILES = {
"src/__init__.py": "",
"src/data/__init__.py": "",
"src/data/select_subset.py": "\"\"\"Fase 3: política de etiquetas, selección de series, muestreo y split del subconjunto.\n\nSolo usa pandas/numpy, así que se prueba en el Mac sin imágenes. El notebook\n`02_subset_rsna_knee` incluye este archivo tal cual.\n\"\"\"\nimport numpy as np\nimport pandas as pd\n\nLABELS = [\n    \"ACL\", \"MCL\", \"Medial Meniscus\", \"Lateral Meniscus\", \"Medial OA\", \"Lateral OA\",\n    \"PF OA\", \"Effusion\", \"Synovitis\", \"Baker's\", \"Contusion\", \"Fracture\",\n]\n\n# Política fijada en la Fase 2 antes de abrir test: solo POS es 1.\n# OJO: en la columna `<label>` de weak_labels.csv MILD figura como 1; por eso se usa `__state`.\nSTATE_TO_Y = {\"POS\": 1, \"MILD\": 0, \"MIN\": 0, \"NEG\": 0, \"NONE\": 0, \"UNC\": -1, \"WEAK\": -1}\n\n# Roles de serie: (plano, combinación preferida, respaldo) con combinación = (fluid, FS).\nROLES = {\"sag\": (\"Sagittal\", (1, 1), (0, 0)), \"cor\": (\"Coronal\", (1, 1), (0, 0))}\nTHIN_MM = 1.5      # por debajo se considera adquisición 3D\nTARGET_SLICES = 30\n\n\ndef apply_policy(weak):\n    \"\"\"weak_labels.csv → una fila por estudio con y_, mask_, conf_, score_ y state_ por etiqueta.\n\n    y_ vale 1/0 (los -1 quedan como 0) y mask_ vale 0 donde el estado es UNC o WEAK.\n    \"\"\"\n    out = {\"StudyInstanceUID\": weak[\"StudyInstanceUID\"].values}\n    for lab in LABELS:\n        state = weak[f\"{lab}__state\"]\n        unknown = set(state.unique()) - set(STATE_TO_Y)\n        if unknown:\n            raise ValueError(f\"{lab}: estados desconocidos {unknown}\")\n        y = state.map(STATE_TO_Y)\n        out[f\"y_{lab}\"] = y.clip(lower=0).astype(int).values\n        out[f\"mask_{lab}\"] = (y != -1).astype(int).values\n        out[f\"conf_{lab}\"] = weak[f\"{lab}__conf\"].values\n        out[f\"score_{lab}\"] = weak[f\"{lab}__score\"].values\n        out[f\"state_{lab}\"] = state.values\n    return pd.DataFrame(out)\n\n\ndef _pick(cands):\n    \"\"\"Entre varias series candidatas: 2D primero, luego cortes cerca de 30, luego mayor matriz.\"\"\"\n    c = cands.assign(\n        _is2d=(cands[\"SliceThickness\"].fillna(0) >= 2).astype(int),\n        _dist=(cands[\"n_slices\"] - TARGET_SLICES).abs(),\n    )\n    c = c.sort_values([\"_is2d\", \"_dist\", \"Rows\"], ascending=[False, True, False])\n    return c.iloc[0]\n\n\ndef choose_series(catalog):\n    \"\"\"series_catalog.csv → una fila por estudio con la serie elegida para cada rol.\n\n    Columnas por rol: <rol>_series, <rol>_fallback (1 si se usó fluid=0 FS=0),\n    <rol>_n_slices, <rol>_thick, <rol>_is3d.\n    \"\"\"\n    rows = []\n    for uid, g in catalog.groupby(\"StudyInstanceUID\", sort=False):\n        r = {\"StudyInstanceUID\": uid}\n        for role, (plane, pref, back) in ROLES.items():\n            pick, fallback = None, 0\n            for i, (fl, fs) in enumerate((pref, back)):\n                m = g[(g[\"Anatomical_Plane\"] == plane) & (g[\"Fluid_Sensitive\"] == fl)\n                      & (g[\"Fat_Suppression\"] == fs) & (g[\"n_slices\"] > 0)]\n                if len(m):\n                    pick, fallback = _pick(m), i\n                    break\n            if pick is None:\n                r.update({f\"{role}_series\": None, f\"{role}_fallback\": np.nan,\n                          f\"{role}_n_slices\": np.nan, f\"{role}_thick\": np.nan, f\"{role}_is3d\": np.nan})\n            else:\n                th = pick[\"SliceThickness\"]\n                r.update({f\"{role}_series\": pick[\"SeriesInstanceUID\"], f\"{role}_fallback\": fallback,\n                          f\"{role}_n_slices\": int(pick[\"n_slices\"]), f\"{role}_thick\": th,\n                          f\"{role}_is3d\": int(pd.notna(th) and th < THIN_MM)})\n        g0 = g.iloc[0]\n        r[\"Manufacturer\"] = g0.get(\"Manufacturer\")\n        r[\"FieldStrength\"] = g0.get(\"FieldStrength\")\n        rows.append(r)\n    return pd.DataFrame(rows)\n\n\ndef eligible_for_training(series):\n    \"\"\"Estudios no dorados utilizables: sagital y coronal presentes y ninguna 3D.\"\"\"\n    s = series\n    ok = s[\"sag_series\"].notna() & s[\"cor_series\"].notna() & (s[\"sag_is3d\"] == 0) & (s[\"cor_is3d\"] == 0)\n    return set(s.loc[ok, \"StudyInstanceUID\"])\n\n\ndef sample_studies(labels, eligible, n=500, min_pos=60, seed=42):\n    \"\"\"Muestreo estratificado: primero asegura `min_pos` positivos (POS) por etiqueta,\n    empezando por la más rara, y completa hasta `n` al azar.\n\n    Devuelve la lista de UIDs elegidos (orden reproducible).\n    \"\"\"\n    rng = np.random.default_rng(seed)\n    pool = labels[labels[\"StudyInstanceUID\"].isin(eligible)].set_index(\"StudyInstanceUID\")\n    pool = pool.loc[rng.permutation(pool.index.values)]\n    chosen = []\n    chosen_set = set()\n    order = sorted(LABELS, key=lambda lab: pool[f\"y_{lab}\"].sum())\n    for lab in order:\n        have = int(pool.loc[chosen, f\"y_{lab}\"].sum()) if chosen else 0\n        if have >= min_pos:\n            continue\n        cands = [u for u in pool.index[pool[f\"y_{lab}\"] == 1] if u not in chosen_set]\n        for u in cands[: min_pos - have]:\n            chosen.append(u)\n            chosen_set.add(u)\n    if len(chosen) > n:\n        raise ValueError(f\"min_pos={min_pos} exige {len(chosen)} estudios > n={n}\")\n    rest = [u for u in pool.index if u not in chosen_set]\n    chosen += rest[: n - len(chosen)]\n    return chosen\n\n\ndef split_train_val(labels, uids, val_frac=0.2, seed=42):\n    \"\"\"Split por estudio. Reparte los positivos de cada etiqueta (de la más rara a la más\n    común) para que val tenga cerca de `val_frac` de ellos. Devuelve {uid: 'train'|'val'}.\"\"\"\n    rng = np.random.default_rng(seed)\n    df = labels.set_index(\"StudyInstanceUID\").loc[list(uids)]\n    df = df.loc[rng.permutation(df.index.values)]\n    n_val = int(round(val_frac * len(df)))\n    split = {}\n    order = sorted(LABELS, key=lambda lab: df[f\"y_{lab}\"].sum())\n    for lab in order:\n        pos = df.index[df[f\"y_{lab}\"] == 1]\n        target = int(round(val_frac * len(pos)))\n        have = sum(split.get(u) == \"val\" for u in pos)\n        for u in pos:\n            if u in split:\n                continue\n            if have < target and sum(v == \"val\" for v in split.values()) < n_val:\n                split[u] = \"val\"\n                have += 1\n            else:\n                split[u] = \"train\"\n    n_now = sum(v == \"val\" for v in split.values())\n    for u in df.index:\n        if u not in split:\n            split[u] = \"val\" if n_now < n_val else \"train\"\n            n_now += split[u] == \"val\"\n    return split\n\n\ndef prevalence(labels, uids, by=None):\n    \"\"\"Positivos y % por etiqueta (y_ == 1) sobre los estudios dados; opcionalmente por grupo.\"\"\"\n    df = labels[labels[\"StudyInstanceUID\"].isin(set(uids))]\n    def one(d):\n        return pd.Series({**{lab: int(d[f\"y_{lab}\"].sum()) for lab in LABELS}, \"n\": len(d)})\n    if by is None:\n        t = one(df).to_frame(\"pos\")\n        t[\"pct\"] = (t[\"pos\"] / len(df) * 100).round(1)\n        t.loc[\"n\", \"pct\"] = np.nan\n        return t\n    return df.groupby(by).apply(one)\n",
"src/models/__init__.py": "",
"src/models/cnn_scratch.py": "\"\"\"Modelo 1: CNN 2D propia que codifica un corte 2.5D (3 cortes contiguos como canales).\n\nBloques conv-BN-ReLU con atajo residual, cuatro etapas con stride 2 y GAP.\nImplementada desde cero (sin torchvision) y entrenada sin pesos previos.\n\"\"\"\nimport torch\nfrom torch import nn\n\n\nclass ConvBNReLU(nn.Sequential):\n    def __init__(self, c_in, c_out, k=3, stride=1):\n        super().__init__(\n            nn.Conv2d(c_in, c_out, k, stride, k // 2, bias=False),\n            nn.BatchNorm2d(c_out),\n            nn.ReLU(inplace=True),\n        )\n\n\nclass ResBlock(nn.Module):\n    \"\"\"Dos convoluciones 3×3 con atajo; el atajo usa conv 1×1 cuando cambia la forma.\"\"\"\n\n    def __init__(self, c_in, c_out, stride=1):\n        super().__init__()\n        self.conv1 = ConvBNReLU(c_in, c_out, 3, stride)\n        self.conv2 = nn.Sequential(nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False), nn.BatchNorm2d(c_out))\n        self.short = nn.Identity()\n        if stride != 1 or c_in != c_out:\n            self.short = nn.Sequential(nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))\n        self.act = nn.ReLU(inplace=True)\n\n    def forward(self, x):\n        return self.act(self.conv2(self.conv1(x)) + self.short(x))\n\n\nclass ScratchCNN(nn.Module):\n    \"\"\"(N, 3, H, W) → (N, out_dim). Con 256×256 la última etapa queda en 8×8.\"\"\"\n\n    def __init__(self, in_ch=3, widths=(32, 64, 128, 256), blocks_per_stage=2):\n        super().__init__()\n        self.stem = nn.Sequential(ConvBNReLU(in_ch, widths[0], 5, 2), ConvBNReLU(widths[0], widths[0]))\n        stages, c = [], widths[0]\n        for w in widths:\n            blocks = [ResBlock(c, w, stride=2)] + [ResBlock(w, w) for _ in range(blocks_per_stage - 1)]\n            stages.append(nn.Sequential(*blocks))\n            c = w\n        self.stages = nn.Sequential(*stages)\n        self.pool = nn.AdaptiveAvgPool2d(1)\n        self.out_dim = c\n        self.cam_layer = self.stages[-1]  # capa objetivo para Grad-CAM\n        for m in self.modules():\n            if isinstance(m, nn.Conv2d):\n                nn.init.kaiming_normal_(m.weight, mode=\"fan_out\", nonlinearity=\"relu\")\n            elif isinstance(m, nn.BatchNorm2d):\n                nn.init.ones_(m.weight)\n                nn.init.zeros_(m.bias)\n\n    def forward(self, x):\n        return torch.flatten(self.pool(self.stages(self.stem(x))), 1)\n",
"src/models/backbone_tl.py": "\"\"\"Modelo 2: backbone de torchvision preentrenado en ImageNet, sin su capa de clasificación.\n\nSolo se toman los pesos del extractor. La agregación por cortes y por plano y la cabeza\nde 12 salidas están en `mil.py` y son propias.\n\"\"\"\nimport torch\nfrom torch import nn\nfrom torchvision import models as tvm\n\nBACKBONES = {\n    \"resnet18\": (tvm.resnet18, 512),\n    \"resnet34\": (tvm.resnet34, 512),\n    \"resnet50\": (tvm.resnet50, 2048),\n    \"efficientnet_b0\": (tvm.efficientnet_b0, 1280),\n}\n\n\nclass TVBackbone(nn.Module):\n    \"\"\"(N, 3, H, W) → (N, out_dim).\"\"\"\n\n    def __init__(self, name=\"resnet18\", pretrained=True):\n        super().__init__()\n        if name not in BACKBONES:\n            raise ValueError(f\"backbone desconocido {name}; opciones {list(BACKBONES)}\")\n        fn, self.out_dim = BACKBONES[name]\n        net = fn(weights=\"DEFAULT\" if pretrained else None)\n        if name.startswith(\"resnet\"):\n            self.features = nn.Sequential(net.conv1, net.bn1, net.relu, net.maxpool,\n                                          net.layer1, net.layer2, net.layer3, net.layer4)\n        else:\n            self.features = net.features\n        self.cam_layer = self.features[-1]\n        self.pool = nn.AdaptiveAvgPool2d(1)\n\n    def forward(self, x):\n        return torch.flatten(self.pool(self.features(x)), 1)\n",
"src/models/mil.py": "\"\"\"Modelo a nivel de estudio: codificador 2D compartido por cortes, agregación por plano y cabeza.\n\nEntrada por plano: (B, D, 3, H, W), es decir D ventanas 2.5D de 3 cortes contiguos.\nCada ventana pasa por el mismo codificador, los D vectores se agregan (max, media o\natención, como en MRNet y en el MIL con atención de Ilse et al.), los planos se\nconcatenan y una capa lineal da 12 logits (una sigmoide independiente por hallazgo).\n\"\"\"\nimport torch\nfrom torch import nn\n\nfrom src.models.cnn_scratch import ScratchCNN\n\n\nclass SlicePool(nn.Module):\n    def __init__(self, dim, mode=\"attn\", hidden=128):\n        super().__init__()\n        self.mode = mode\n        if mode == \"attn\":\n            self.att = nn.Sequential(nn.Linear(dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))\n        elif mode not in (\"max\", \"mean\"):\n            raise ValueError(f\"pool desconocido {mode}\")\n\n    def forward(self, h):\n        \"\"\"h: (B, D, dim) → (B, dim), pesos por corte (B, D) o None.\"\"\"\n        if self.mode == \"max\":\n            return h.max(1).values, None\n        if self.mode == \"mean\":\n            return h.mean(1), None\n        a = torch.softmax(self.att(h).squeeze(-1).float(), dim=1).to(h.dtype)\n        return (a.unsqueeze(-1) * h).sum(1), a\n\n\ndef make_encoder(cfg):\n    if cfg[\"model\"] == \"scratch\":\n        return ScratchCNN(widths=tuple(cfg.get(\"widths\", (32, 64, 128, 256))))\n    from src.models.backbone_tl import TVBackbone  # torchvision solo hace falta aquí\n    return TVBackbone(cfg[\"model\"], pretrained=cfg.get(\"pretrained\", True))\n\n\nclass StudyNet(nn.Module):\n    def __init__(self, cfg, n_out=12):\n        super().__init__()\n        self.planes = list(cfg[\"planes\"])\n        self.encoders = nn.ModuleDict({p: make_encoder(cfg) for p in self.planes})\n        dim = next(iter(self.encoders.values())).out_dim\n        self.pools = nn.ModuleDict({p: SlicePool(dim, cfg.get(\"pool\", \"attn\")) for p in self.planes})\n        self.head = nn.Sequential(nn.Dropout(cfg.get(\"dropout\", 0.3)), nn.Linear(dim * len(self.planes), n_out))\n\n    def forward(self, x, return_attn=False):\n        feats, attn = [], {}\n        for p in self.planes:\n            v = x[p]\n            b, d = v.shape[:2]\n            h = self.encoders[p](v.flatten(0, 1)).view(b, d, -1)\n            z, a = self.pools[p](h)\n            feats.append(z)\n            attn[p] = a\n        logits = self.head(torch.cat(feats, 1))\n        return (logits, attn) if return_attn else logits\n\n    def backbone_parameters(self):\n        return [q for e in self.encoders.values() for q in e.parameters()]\n\n    def set_backbone_trainable(self, flag):\n        \"\"\"Congela o libera los codificadores. Congelados, su BN queda en modo eval.\"\"\"\n        for e in self.encoders.values():\n            for q in e.parameters():\n                q.requires_grad_(flag)\n            e.train(flag and self.training)\n        self._bb_frozen = not flag\n\n    def train(self, mode=True):\n        super().train(mode)\n        if getattr(self, \"_bb_frozen\", False):\n            for e in self.encoders.values():\n                e.eval()\n        return self\n",
"src/train/__init__.py": "",
"src/train/data.py": "\"\"\"Fase 4: lectura de rsna-knee-subset-v1 y armado de ventanas 2.5D por estudio.\n\nCada volumen (n, 256, 256) uint8 se reduce a D ventanas de `window` cortes contiguos\n(los cortes van como canales). Los centros se reparten de forma uniforme en la serie;\nen entrenamiento se desplazan todos juntos al azar como aumento.\n\nObjetivos por split:\n- train y val: y_ (política POS = 1) con pesos según la estrategia de ruido.\n- test: los 58 dorados (gold_*), todos con peso 1.\n\"\"\"\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom src.data.select_subset import LABELS\n\n\ndef load_studies(data_dir):\n    df = pd.read_csv(Path(data_dir) / \"subset_studies.csv\")\n    assert df.StudyInstanceUID.is_unique\n    return df\n\n\ndef label_weights(df, noise=\"mask\", highconf_thr=0.9):\n    \"\"\"Peso por estudio y etiqueta (N, 12) según la estrategia de manejo de ruido.\n\n    - mask: 1 donde la etiqueta es conocida (UNC y WEAK quedan en 0).\n    - conf: la confianza del extractor (POS 0.9 a 0.95, NEG 0.9, MIN 0.7, NONE y MILD 0.6).\n    - highconf: 1 solo si conf >= highconf_thr; con 0.9 quedan POS y NEG explícitos y se\n      descartan los \"no mencionados\".\n    \"\"\"\n    mask = df[[f\"mask_{l}\" for l in LABELS]].to_numpy(np.float32)\n    conf = df[[f\"conf_{l}\" for l in LABELS]].to_numpy(np.float32)\n    if noise == \"mask\":\n        return mask\n    if noise == \"conf\":\n        return mask * conf\n    if noise == \"highconf\":\n        return mask * (conf >= highconf_thr - 1e-6)\n    raise ValueError(f\"noise desconocido {noise}\")\n\n\ndef targets(df, split, noise=\"mask\", highconf_thr=0.9):\n    \"\"\"(y, w) como arreglos (N, 12) float32 para las filas de `df` (todas del mismo split).\"\"\"\n    if split == \"test\":\n        y = df[[f\"gold_{l}\" for l in LABELS]].to_numpy(np.float32)\n        assert not np.isnan(y).any(), \"test sin etiqueta dorada\"\n        return y, np.ones_like(y)\n    y = df[[f\"y_{l}\" for l in LABELS]].to_numpy(np.float32)\n    return y, label_weights(df, noise, highconf_thr)\n\n\ndef window_indices(n, n_windows=16, window=3, jitter=0.0, rng=None):\n    \"\"\"Índices (n_windows, window) de cortes. jitter en fracción del paso entre centros.\"\"\"\n    half = window // 2\n    lo, hi = half, max(half, n - 1 - half)\n    centers = np.linspace(lo, hi, n_windows)\n    if jitter and rng is not None:\n        step = (hi - lo) / max(n_windows - 1, 1)\n        centers = centers + rng.uniform(-jitter, jitter) * step\n    centers = np.clip(np.round(centers), lo, hi).astype(int)\n    idx = centers[:, None] + np.arange(-half, half + 1)[None, :]\n    return np.clip(idx, 0, n - 1)\n\n\nclass KneeStudies(Dataset):\n    \"\"\"Devuelve {\"x\": {plano: uint8 (D, window, H, W)}, \"y\": (12,), \"w\": (12,), \"i\": índice}.\"\"\"\n\n    PLANE_FILE = {\"sag\": \"sag\", \"cor\": \"cor\"}\n\n    def __init__(self, df, data_dir, split, planes=(\"sag\",), n_windows=16, window=3,\n                 noise=\"mask\", highconf_thr=0.9, train=False, cache=None, seed=0):\n        self.df = df.reset_index(drop=True)\n        self.dir = Path(data_dir)\n        self.planes, self.n_windows, self.window, self.train = list(planes), n_windows, window, train\n        self.y, self.w = targets(self.df, split, noise, highconf_thr)\n        self.cache = {} if cache is None else cache\n        self.rng = np.random.default_rng(seed)\n        for uid in self.df.StudyInstanceUID:\n            for p in self.planes:\n                key = f\"{uid}_{self.PLANE_FILE[p]}\"\n                if key not in self.cache:\n                    self.cache[key] = np.load(self.dir / \"vol\" / f\"{key}.npy\")\n\n    def __len__(self):\n        return len(self.df)\n\n    def __getitem__(self, i):\n        uid = self.df.StudyInstanceUID.iat[i]\n        x = {}\n        for p in self.planes:\n            v = self.cache[f\"{uid}_{self.PLANE_FILE[p]}\"]\n            idx = window_indices(len(v), self.n_windows, self.window,\n                                 jitter=0.5 if self.train else 0.0, rng=self.rng)\n            x[p] = torch.from_numpy(np.ascontiguousarray(v[idx]))\n        return {\"x\": x, \"y\": torch.from_numpy(self.y[i]), \"w\": torch.from_numpy(self.w[i]), \"i\": i}\n\n\ndef worker_init(worker_id):\n    info = torch.utils.data.get_worker_info()\n    info.dataset.rng = np.random.default_rng(torch.initial_seed() % 2**32)\n",
"src/train/augment.py": "\"\"\"Aumentos en GPU, iguales para todos los cortes de un mismo estudio y plano.\n\nSin flips: en sagital el flip horizontal cambia anterior por posterior y en coronal\ncambia medial por lateral, lo que alteraría Medial y Lateral Meniscus/OA.\nSe usan rotación leve, escala, traslación, contraste, brillo y gamma.\n\"\"\"\nimport math\n\nimport torch\nimport torch.nn.functional as F\n\nMEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 1, 3, 1, 1)\nSTD = torch.tensor([0.229, 0.224, 0.225]).view(1, 1, 3, 1, 1)\n\n\ndef to_float(x):\n    \"\"\"uint8 (B, D, 3, H, W) → float en [0, 1].\"\"\"\n    return x.float().div_(255)\n\n\ndef normalize(x):\n    \"\"\"Estandarización con la media y desviación de ImageNet (para ambos modelos).\"\"\"\n    return (x - MEAN.to(x.device)) / STD.to(x.device)\n\n\ndef random_affine(x, max_deg=10, scale=(0.9, 1.1), max_shift=0.06):\n    b, d, c, h, w = x.shape\n    dev = x.device\n    ang = (torch.rand(b, device=dev) * 2 - 1) * math.radians(max_deg)\n    s = torch.empty(b, device=dev).uniform_(*scale)\n    t = (torch.rand(b, 2, device=dev) * 2 - 1) * max_shift * 2\n    cos, sin = torch.cos(ang) / s, torch.sin(ang) / s\n    theta = torch.stack([torch.stack([cos, -sin, t[:, 0]], 1), torch.stack([sin, cos, t[:, 1]], 1)], 1)\n    theta = theta.repeat_interleave(d, 0)\n    flat = x.reshape(b * d, c, h, w)\n    grid = F.affine_grid(theta, flat.shape, align_corners=False)\n    return F.grid_sample(flat, grid, mode=\"bilinear\", padding_mode=\"zeros\", align_corners=False).view(b, d, c, h, w)\n\n\ndef random_intensity(x, contrast=(0.8, 1.2), brightness=0.1, gamma=(0.8, 1.25)):\n    b = x.shape[0]\n    shape = (b, 1, 1, 1, 1)\n    dev = x.device\n    g = torch.exp(torch.empty(shape, device=dev).uniform_(math.log(gamma[0]), math.log(gamma[1])))\n    x = x.clamp(0, 1).pow(g)\n    k = torch.empty(shape, device=dev).uniform_(*contrast)\n    m = x.mean(dim=(1, 2, 3, 4), keepdim=True)\n    x = (x - m) * k + m + torch.empty(shape, device=dev).uniform_(-brightness, brightness)\n    return x.clamp(0, 1)\n\n\ndef prepare(x, train):\n    \"\"\"uint8 en el dispositivo → tensor normalizado, con aumentos si train.\"\"\"\n    x = to_float(x)\n    if train:\n        x = random_intensity(random_affine(x))\n    return normalize(x)\n",
"src/train/losses.py": "\"\"\"Pérdida multietiqueta con pesos por elemento (máscara o confianza).\n\n- bce: entropía cruzada binaria, con pos_weight opcional para los hallazgos raros.\n- focal: la misma BCE multiplicada por (1 - p_t)^gamma (Lin et al., 2017).\n\"\"\"\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\n\n\ndef auto_pos_weight(y, w, max_pw=10.0):\n    \"\"\"neg / pos por etiqueta contando solo los elementos con peso > 0, acotado a [1, max_pw].\"\"\"\n    known = w > 0\n    pos = ((y == 1) & known).sum(0).astype(np.float32)\n    neg = ((y == 0) & known).sum(0).astype(np.float32)\n    return np.clip(neg / np.maximum(pos, 1), 1.0, max_pw).astype(np.float32)\n\n\ndef multilabel_loss(logits, y, w, kind=\"bce\", pos_weight=None, gamma=2.0):\n    logits = logits.float()\n    loss = F.binary_cross_entropy_with_logits(logits, y, reduction=\"none\", pos_weight=pos_weight)\n    if kind == \"focal\":\n        p = torch.sigmoid(logits)\n        pt = p * y + (1 - p) * (1 - y)\n        loss = loss * (1 - pt).pow(gamma)\n    elif kind != \"bce\":\n        raise ValueError(f\"loss desconocida {kind}\")\n    return (loss * w).sum() / w.sum().clamp_min(1.0)\n",
"src/train/metrics.py": "\"\"\"AUC-ROC por etiqueta y macro (métrica oficial), ignorando elementos con peso 0.\"\"\"\nimport numpy as np\nfrom sklearn.metrics import roc_auc_score\n\nfrom src.data.select_subset import LABELS\n\n\ndef auc_table(y, p, w=None):\n    \"\"\"Devuelve ({etiqueta: AUC o NaN}, AUC macro). NaN si la etiqueta tiene una sola clase.\"\"\"\n    w = np.ones_like(y) if w is None else w\n    out = {}\n    for j, lab in enumerate(LABELS):\n        m = w[:, j] > 0\n        yj = y[m, j]\n        out[lab] = float(roc_auc_score(yj, p[m, j])) if 0 < yj.sum() < len(yj) else float(\"nan\")\n    vals = [v for v in out.values() if not np.isnan(v)]\n    return out, float(np.mean(vals)) if vals else float(\"nan\")\n",
"src/train/train.py": "\"\"\"Fase 4: entrena un modelo a nivel de estudio y guarda checkpoint, predicciones y metrics.json.\n\nUso (desde rsna-knee-weak/):\n    python -m src.train.train --config configs/scratch_base.json --data <dir del subset> --out runs/\n    python -m src.train.train --config configs/resnet18_base.json --data ... --set epochs=1 limit_train=16\n\nSelección del checkpoint: AUC macro en val (etiquetas débiles). Los 58 dorados (test) solo se\npredicen al final con el mejor checkpoint y nunca intervienen en el entrenamiento ni en la selección.\n\"\"\"\nimport argparse\nimport json\nimport math\nimport platform\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom src.data.select_subset import LABELS\nfrom src.models.mil import StudyNet\nfrom src.train.augment import prepare\nfrom src.train.data import KneeStudies, load_studies, worker_init\nfrom src.train.losses import auto_pos_weight, multilabel_loss\nfrom src.train.metrics import auc_table\n\nDEFAULTS = {\n    \"name\": \"run\",\n    \"model\": \"scratch\",          # scratch | resnet18 | resnet34 | resnet50 | efficientnet_b0\n    \"pretrained\": True,\n    \"planes\": [\"sag\", \"cor\"],\n    \"n_windows\": 16, \"window\": 3,\n    \"pool\": \"attn\",              # attn | max | mean\n    \"loss\": \"bce\", \"focal_gamma\": 2.0,\n    \"pos_weight\": \"auto\", \"max_pos_weight\": 10.0,   # auto | none\n    \"noise\": \"mask\", \"highconf_thr\": 0.9,           # mask | conf | highconf\n    \"epochs\": 30, \"batch_size\": 4, \"lr\": 1e-3, \"bb_lr_mult\": 1.0, \"freeze_epochs\": 0,\n    \"weight_decay\": 1e-4, \"warmup_epochs\": 1, \"dropout\": 0.3, \"patience\": 8,\n    \"aug\": True, \"amp\": True, \"num_workers\": 2, \"seed\": 42,\n    \"limit_train\": 0, \"limit_eval\": 0,              # >0 solo para pruebas rápidas\n}\n\n\ndef parse_value(s):\n    try:\n        return json.loads(s)\n    except json.JSONDecodeError:\n        return s\n\n\ndef load_config(path, overrides):\n    cfg = dict(DEFAULTS)\n    if path:\n        cfg.update(json.loads(Path(path).read_text()))\n    for kv in overrides or []:\n        k, v = kv.split(\"=\", 1)\n        if k not in DEFAULTS:\n            raise KeyError(f\"clave desconocida {k}\")\n        cfg[k] = parse_value(v)\n    return cfg\n\n\ndef pick_device():\n    if torch.cuda.is_available():\n        return torch.device(\"cuda\")\n    if torch.backends.mps.is_available():\n        return torch.device(\"mps\")\n    return torch.device(\"cpu\")\n\n\ndef to_device(batch, dev):\n    x = {p: v.to(dev, non_blocking=True) for p, v in batch[\"x\"].items()}\n    return x, batch[\"y\"].to(dev), batch[\"w\"].to(dev)\n\n\n@torch.no_grad()\ndef predict(model, loader, dev, amp):\n    model.eval()\n    probs, idx = [], []\n    for batch in loader:\n        x, _, _ = to_device(batch, dev)\n        with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):\n            logits = model({p: prepare(v, train=False) for p, v in x.items()})\n        probs.append(torch.sigmoid(logits.float()).cpu().numpy())\n        idx.append(batch[\"i\"].numpy())\n    order = np.argsort(np.concatenate(idx))\n    return np.concatenate(probs)[order]\n\n\ndef save_preds(path, ds, p):\n    df = pd.DataFrame(p, columns=[f\"p_{l}\" for l in LABELS])\n    df.insert(0, \"StudyInstanceUID\", ds.df.StudyInstanceUID.values)\n    for j, l in enumerate(LABELS):\n        df[f\"y_{l}\"] = ds.y[:, j]\n        df[f\"w_{l}\"] = ds.w[:, j]\n    for c in (\"lang\", \"Manufacturer\", \"gold_split\"):\n        if c in ds.df:\n            df[c] = ds.df[c].values\n    df.to_csv(path, index=False)\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--config\")\n    ap.add_argument(\"--data\", required=True, help=\"carpeta con subset_studies.csv y vol/\")\n    ap.add_argument(\"--out\", default=\"runs\")\n    ap.add_argument(\"--set\", nargs=\"*\", default=[], help=\"sobrescribe claves: epochs=1 lr=3e-4\")\n    args = ap.parse_args(argv)\n    cfg = load_config(args.config, args.set)\n\n    torch.manual_seed(cfg[\"seed\"])\n    np.random.seed(cfg[\"seed\"])\n    dev = pick_device()\n    amp = bool(cfg[\"amp\"]) and dev.type == \"cuda\"\n    out = Path(args.out) / cfg[\"name\"]\n    out.mkdir(parents=True, exist_ok=True)\n    print(f\"dispositivo {dev} · amp {amp} · salida {out}\")\n    print(json.dumps(cfg))\n\n    studies = load_studies(args.data)\n    parts = {s: studies[studies.split == s] for s in (\"train\", \"val\", \"test\")}\n    if cfg[\"limit_train\"]:\n        parts[\"train\"] = parts[\"train\"].sample(min(cfg[\"limit_train\"], len(parts[\"train\"])), random_state=0)\n    if cfg[\"limit_eval\"]:\n        for s in (\"val\", \"test\"):\n            parts[s] = parts[s].sample(min(cfg[\"limit_eval\"], len(parts[s])), random_state=0)\n    common = dict(data_dir=args.data, planes=cfg[\"planes\"], n_windows=cfg[\"n_windows\"], window=cfg[\"window\"],\n                  noise=cfg[\"noise\"], highconf_thr=cfg[\"highconf_thr\"])\n    cache = {}\n    t0 = time.time()\n    ds = {s: KneeStudies(parts[s], split=s, train=(s == \"train\" and cfg[\"aug\"]), cache=cache,\n                         seed=cfg[\"seed\"], **common) for s in parts}\n    print({s: len(d) for s, d in ds.items()}, f\"· volúmenes en memoria: {len(cache)} en {time.time()-t0:.0f} s\")\n\n    lk = dict(num_workers=cfg[\"num_workers\"], pin_memory=dev.type == \"cuda\",\n              persistent_workers=cfg[\"num_workers\"] > 0, worker_init_fn=worker_init)\n    dl_train = DataLoader(ds[\"train\"], batch_size=cfg[\"batch_size\"], shuffle=True, drop_last=True, **lk)\n    dl_eval = {s: DataLoader(ds[s], batch_size=cfg[\"batch_size\"], shuffle=False, **lk) for s in (\"val\", \"test\")}\n\n    pw = None\n    if cfg[\"pos_weight\"] == \"auto\":\n        pw_np = auto_pos_weight(ds[\"train\"].y, ds[\"train\"].w, cfg[\"max_pos_weight\"])\n        pw = torch.tensor(pw_np, device=dev)\n        print(\"pos_weight:\", dict(zip(LABELS, pw_np.round(2).tolist())))\n\n    model = StudyNet(cfg, n_out=len(LABELS)).to(dev)\n    if dev.type == \"cuda\":\n        model = model.to(memory_format=torch.channels_last)\n    bb = set(map(id, model.backbone_parameters()))\n    head = [q for q in model.parameters() if id(q) not in bb]\n    opt = torch.optim.AdamW([\n        {\"params\": model.backbone_parameters(), \"lr\": cfg[\"lr\"] * cfg[\"bb_lr_mult\"]},\n        {\"params\": head, \"lr\": cfg[\"lr\"]},\n    ], weight_decay=cfg[\"weight_decay\"])\n    base_lrs = [g[\"lr\"] for g in opt.param_groups]\n    steps_per_epoch = len(dl_train)\n    total, warm = cfg[\"epochs\"] * steps_per_epoch, cfg[\"warmup_epochs\"] * steps_per_epoch\n\n    def lr_factor(step):\n        if step < warm:\n            return (step + 1) / warm\n        return 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))\n\n    scaler = torch.amp.GradScaler(\"cuda\", enabled=amp)\n    n_params = sum(q.numel() for q in model.parameters())\n    print(f\"modelo {cfg['model']} · {n_params/1e6:.2f} M parámetros · {steps_per_epoch} pasos por época\")\n\n    best, best_ep, bad, hist, step = -1.0, -1, 0, [], 0\n    t_start = time.time()\n    for ep in range(cfg[\"epochs\"]):\n        frozen = ep < cfg[\"freeze_epochs\"]\n        model.set_backbone_trainable(not frozen)\n        model.train()\n        te, run_loss = time.time(), 0.0\n        for batch in dl_train:\n            x, y, w = to_device(batch, dev)\n            for g, b in zip(opt.param_groups, base_lrs):\n                g[\"lr\"] = b * lr_factor(step)\n            with torch.autocast(dev.type, dtype=torch.float16, enabled=amp):\n                logits = model({p: prepare(v, train=cfg[\"aug\"]) for p, v in x.items()})\n            loss = multilabel_loss(logits, y, w, cfg[\"loss\"], pw, cfg[\"focal_gamma\"])\n            opt.zero_grad(set_to_none=True)\n            scaler.scale(loss).backward()\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)\n            scaler.step(opt)\n            scaler.update()\n            run_loss += loss.item()\n            step += 1\n        p_val = predict(model, dl_eval[\"val\"], dev, amp)\n        per, macro = auc_table(ds[\"val\"].y, p_val, ds[\"val\"].w)\n        hist.append({\"epoch\": ep, \"loss\": run_loss / max(steps_per_epoch, 1), \"val_auc\": macro,\n                     \"frozen\": int(frozen), \"sec\": time.time() - te, **{f\"val_{k}\": v for k, v in per.items()}})\n        flag = \"\"\n        if macro > best:\n            best, best_ep, bad, flag = macro, ep, 0, \" *\"\n            torch.save({\"cfg\": cfg, \"state_dict\": model.state_dict(), \"epoch\": ep}, out / \"best.pt\")\n        else:\n            bad += 1\n        print(f\"ep {ep:02d} · loss {hist[-1]['loss']:.4f} · val AUC {macro:.4f}{flag} · {hist[-1]['sec']:.0f} s\"\n              + (\" · backbone congelado\" if frozen else \"\"), flush=True)\n        pd.DataFrame(hist).to_csv(out / \"history.csv\", index=False)\n        if bad >= cfg[\"patience\"] and not frozen:\n            print(f\"parada temprana en la época {ep}\")\n            break\n\n    ck = torch.load(out / \"best.pt\", map_location=dev, weights_only=False)\n    model.load_state_dict(ck[\"state_dict\"])\n    res = {}\n    for s in (\"val\", \"test\"):\n        p = predict(model, dl_eval[s], dev, amp)\n        save_preds(out / f\"preds_{s}.csv\", ds[s], p)\n        res[s] = auc_table(ds[s].y, p, ds[s].w)\n    metrics = {\n        \"name\": cfg[\"name\"], \"cfg\": cfg, \"best_epoch\": best_ep, \"epochs_run\": len(hist),\n        \"val_auc_macro\": res[\"val\"][1], \"test_auc_macro\": res[\"test\"][1],\n        \"val_auc\": res[\"val\"][0], \"test_auc\": res[\"test\"][0],\n        \"n_params\": n_params, \"minutes\": (time.time() - t_start) / 60,\n        \"gpu_peak_gb\": torch.cuda.max_memory_allocated() / 1e9 if dev.type == \"cuda\" else None,\n        \"device\": torch.cuda.get_device_name(0) if dev.type == \"cuda\" else str(dev),\n        \"torch\": torch.__version__, \"python\": platform.python_version(),\n        \"n\": {s: len(d) for s, d in ds.items()},\n    }\n    (out / \"metrics.json\").write_text(json.dumps(metrics, indent=1))\n    if dev.type == \"cuda\":\n        print(f\"memoria GPU máxima {metrics['gpu_peak_gb']:.1f} GB\")\n    print(f\"mejor época {best_ep} · val AUC {res['val'][1]:.4f} · test (58 dorados) AUC {res['test'][1]:.4f}\")\n    print(pd.DataFrame({\"val\": res[\"val\"][0], \"test\": res[\"test\"][0]}).round(3).T.to_string())\n    return metrics\n\n\nif __name__ == \"__main__\":\n    main()\n",
"configs/scratch_base.json": "{\n \"name\": \"scratch_base\",\n \"model\": \"scratch\",\n \"planes\": [\"sag\", \"cor\"],\n \"lr\": 1e-3,\n \"batch_size\": 8,\n \"epochs\": 40,\n \"patience\": 10\n}\n",
"configs/resnet18_base.json": "{\n \"name\": \"resnet18_base\",\n \"model\": \"resnet18\",\n \"pretrained\": true,\n \"planes\": [\"sag\", \"cor\"],\n \"lr\": 3e-4,\n \"bb_lr_mult\": 0.1,\n \"freeze_epochs\": 2,\n \"batch_size\": 4,\n \"epochs\": 25,\n \"patience\": 8\n}\n"
}
for rel, src in FILES.items():
    p = CODE / rel
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(src)
print(len(FILES), "archivos en", CODE)

def train(config, *overrides, out=RUNS):
    cmd = [sys.executable, "-m", "src.train.train", "--config", f"configs/{config}.json",
           "--data", str(DATA), "--out", str(out)]
    if overrides:
        cmd += ["--set", *overrides]
    print(" ".join(cmd), flush=True)
    subprocess.run(cmd, cwd=CODE, check=True)

## 2. Prueba rápida (1 época con 16 estudios; unos 2 min). Si falla, no seguir.

In [ ]:
SMOKE = Path("/tmp/smoke")
train("scratch_base", "epochs=1", "limit_train=16", "limit_eval=16", out=SMOKE)
train("resnet18_base", "epochs=2", "freeze_epochs=1", "limit_train=16", "limit_eval=16", out=SMOKE)

## 3. Modelo 1: CNN desde cero

In [ ]:
train("scratch_base")

## 4. Modelo 2: ResNet-18 preentrenado

In [ ]:
train("resnet18_base")

## 5. Resumen

In [ ]:
import matplotlib.pyplot as plt
rows, fig, ax = [], *plt.subplots(1, 2, figsize=(12, 4))
for m in sorted(RUNS.glob("*/metrics.json")):
    r = json.loads(m.read_text())
    rows.append({"run": r["name"], "best_epoch": r["best_epoch"], "val_auc": r["val_auc_macro"],
                 "test_auc": r["test_auc_macro"], "min": r["minutes"], "M_params": r["n_params"] / 1e6})
    h = pd.read_csv(m.parent / "history.csv")
    ax[0].plot(h.epoch, h.loss, label=r["name"]); ax[1].plot(h.epoch, h.val_auc, label=r["name"])
ax[0].set_title("pérdida de entrenamiento"); ax[1].set_title("AUC macro en val (débil)")
for a in ax: a.set_xlabel("época"); a.legend()
plt.show()
summary = pd.DataFrame(rows).round(4); summary.to_csv(RUNS / "summary.csv", index=False); print(summary)
per = pd.DataFrame({r: json.loads((RUNS / r / "metrics.json").read_text())["test_auc"] for r in summary.run}).round(3)
print("AUC por etiqueta en los 58 dorados"); print(per)

## 6. Guardar

**Save Version → Save & Run All (Commit)**. El entrenamiento corre en el commit, así que la sesión interactiva se puede cerrar.
La versión guardada deja `runs/` como output del notebook; la Fase 5 (evaluación, ablación y Grad-CAM) lo agrega como input,
igual que el 02 usó el output del 01. Es un derivado de los datos: no se publica ni se sube a GitHub.